# Phase 6: Statistical Diagnostics & Feature Quality Audit

## Project: Credit Risk Modelling & Independent Model Validation
**Target Role**: Quantitative Risk Analytics / Credit Risk Model Validation

### Scope of Notebook
- Part 1: Data Preparation & Target Definition
- Part 2: Comprehensive Descriptive Statistics
- Part 3: Normality Diagnostics (Shapiro-Wilk, Jarque-Bera, Anderson-Darling, D'Agostino K²)
- Part 4: Multicollinearity Audit (Correlation, VIF, Condition Index)
- Part 10: Categorical Variable Encoding Strategy
- Part 11: Feature Stability (PSI/CSI)

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))

from validation.statistical_tests import (
    calculate_descriptive_stats,
    run_normality_tests,
    generate_normality_plots,
    calculate_csi,
)
from validation.econometrics import (
    calculate_correlations,
    find_high_correlations,
    calculate_vif_and_tolerance,
    calculate_condition_index_and_vdp,
)

pd.set_option("display.max_columns", 30)
print("Validation modules loaded successfully!")

Validation modules loaded successfully!


In [2]:
data_path = Path.cwd().parent / "data" / "processed" / "accepted_2007_to_2018Q4_feature_engineered.csv.gz"
cols_to_use = [
    "loan_status", "issue_d", "loan_amnt", "int_rate", "installment", "annual_inc",
    "dti", "fico_range_low", "revol_util", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "pub_rec", "revol_bal", "total_acc", "grade", "term",
    "home_ownership", "verification_status", "purpose",
    "fe_loan_to_income_ratio", "fe_monthly_installment_to_income_ratio",
    "fe_credit_utilization", "fe_available_revolving_credit", "fe_credit_exposure",
    "fe_debt_burden", "fe_credit_history_months"
]
df = pd.read_csv(data_path, usecols=cols_to_use, nrows=100000, low_memory=False)

bad_statuses = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]
good_statuses = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
df["target"] = np.nan
df.loc[df["loan_status"].isin(bad_statuses), "target"] = 1.0
df.loc[df["loan_status"].isin(good_statuses), "target"] = 0.0

df_model = df.dropna(subset=["target"]).copy()
print(f"Binary dataset size: {len(df_model):,}, Default Rate: {df_model['target'].mean():.4%}")

Binary dataset size: 88,333, Default Rate: 20.4284%


In [3]:
numeric_features = [
    "loan_amnt", "int_rate", "installment", "annual_inc", "dti", "fico_range_low",
    "revol_util", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
    "total_acc", "fe_loan_to_income_ratio", "fe_monthly_installment_to_income_ratio",
    "fe_credit_utilization", "fe_available_revolving_credit", "fe_credit_exposure",
    "fe_debt_burden", "fe_credit_history_months"
]
desc_table = calculate_descriptive_stats(df_model, numeric_features)
desc_table

,feature,count,mean,median,mode,variance,std_dev,coef_variation,min,max,range,q1,q3,iqr,skewness,kurtosis
0,loan_amnt,88333,14397.9161,12000.0000,10000.0000,7.390442e+07,8596.7681,0.5971,1000.0000,3.500000e+04,3.400000e+04,8000.0000,20000.0000,12000.0000,0.7172,-0.2561
1,int_rate,88333,11.9614,11.5300,9.1700,1.707420e+01,4.1321,0.3455,5.3200,2.899000e+01,2.367000e+01,8.4900,14.3300,5.8400,0.6850,0.4648
2,installment,88333,430.0196,369.9300,318.7900,6.426209e+04,253.4997,0.5895,14.7700,1.354660e+03,1.339890e+03,245.5100,572.1900,326.6800,0.9468,0.5275
3,annual_inc,88333,77688.3436,65000.0000,60000.0000,8.014093e+09,89521.4653,1.1523,0.0000,9.000000e+06,9.000000e+06,46000.0000,92678.0000,46678.0000,52.8508,4499.7725
4,dti,88331,18.9970,18.4200,15.6000,9.406560e+01,9.6987,0.5105,0.0000,9.990000e+02,9.990000e+02,12.3300,25.2300,12.9000,15.9693,1434.8931
5,fico_range_low,88333,694.4741,685.0000,660.0000,9.662862e+02,31.0851,0.0448,660.0000,8.450000e+02,1.850000e+02,670.0000,710.0000,40.0000,1.3522,1.9629
6,revol_util,88297,51.6547,51.4000,0.0000,5.824486e+02,24.1340,0.4672,0.0000,1.525000e+02,1.525000e+02,33.5000,69.9000,36.4000,0.0111,-0.7734
7,delinq_2yrs,88333,0.3505,0.0000,0.0000,8.627000e-01,0.9288,2.6501,0.0000,3.000000e+01,3.000000e+01,0.0000,0.0000,0.0000,5.2274,50.0091
8,inq_last_6mths,88333,0.5988,0.0000,0.0000,7.852000e-01,0.8861,1.4798,0.0000,5.000000e+00,5.000000e+00,0.0000,1.0000,1.0000,1.7223,3.2280
9,open_acc,88333,11.8635,11.0000,9.0000,3.207500e+01,5.6635,0.4774,1.0000,6.700000e+01,6.600000e+01,8.0000,15.0000,7.0000,1.2705,2.9403


In [4]:
norm_results = run_normality_tests(df_model, numeric_features[:10])
norm_results

,feature,shapiro_stat,shapiro_pvalue,jarque_bera_stat,jarque_bera_pvalue,dagostino_stat,dagostino_pvalue,anderson_stat,anderson_crit_5pct,normality_holds_5pct
0,loan_amnt,0.93909,3.109997e-41,7.814830e+03,0.0,3678.93,0.0,86.2492,0.786,No
1,int_rate,0.95993,4.591920e-35,7.702600e+03,0.0,3499.86,0.0,43.0182,0.786,No
2,installment,0.93360,1.434349e-42,1.421944e+04,0.0,5843.00,0.0,91.8777,0.786,No
3,annual_inc,0.57118,7.291358e-77,7.455612e+10,0.0,134636.70,0.0,359.8381,0.786,No
4,dti,0.98826,5.521222e-20,7.580659e+09,0.0,112801.61,0.0,11.1363,0.786,No
5,fico_range_low,0.87253,2.209711e-53,4.109368e+04,0.0,11466.32,0.0,162.7205,0.786,No
6,revol_util,0.98614,9.163656e-22,2.202530e+03,0.0,3867.90,0.0,11.7183,0.786,No
7,delinq_2yrs,0.43593,8.885076e-83,9.605896e+06,0.0,49073.97,0.0,1061.4843,0.786,No
8,inq_last_6mths,0.70718,5.237649e-69,8.201491e+04,0.0,16022.18,0.0,565.3928,0.786,No
9,open_acc,0.92983,1.948417e-43,5.557568e+04,0.0,11904.26,0.0,77.6459,0.786,No


In [5]:
vif_table = calculate_vif_and_tolerance(df_model, numeric_features[:12])
print("VIF & Tolerance:")
print(vif_table)

ci_table, _ = calculate_condition_index_and_vdp(df_model, numeric_features[:10])
print("Condition Index:")
print(ci_table)

VIF & Tolerance:
           feature      vif  tolerance  \
0        loan_amnt  12.3591     0.0809   
1      installment  11.8155     0.0846   
2   fico_range_low   1.6978     0.5890   
3       revol_util   1.5713     0.6364   
4         int_rate   1.4209     0.7038   
5        revol_bal   1.3536     0.7388   
6         open_acc   1.3064     0.7654   
7              dti   1.2137     0.8239   
8       annual_inc   1.1832     0.8452   
9   inq_last_6mths   1.1515     0.8685   
10         pub_rec   1.0937     0.9143   
11     delinq_2yrs   1.0501     0.9523   

                                       recommendation  
0   High collinearity - Recommend removal or aggre...  
1   High collinearity - Recommend removal or aggre...  
2                           Low collinearity - Retain  
3                           Low collinearity - Retain  
4                           Low collinearity - Retain  
5                           Low collinearity - Retain  
6                           Low collinearity

In [6]:
df["year"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce").dt.year
c2015 = df[df["year"] == 2015]
c2018 = df[df["year"] == 2018]
csi_table = calculate_csi(c2015, c2018, numeric_features[:10])
csi_table

,feature,psi_csi_value,stability_status
0,loan_amnt,NaN,Significant Shift (Action Required)
1,int_rate,NaN,Significant Shift (Action Required)
2,installment,NaN,Significant Shift (Action Required)
3,annual_inc,NaN,Significant Shift (Action Required)
4,dti,NaN,Significant Shift (Action Required)
5,fico_range_low,NaN,Significant Shift (Action Required)
6,revol_util,NaN,Significant Shift (Action Required)
7,delinq_2yrs,NaN,Significant Shift (Action Required)
8,inq_last_6mths,8.1684,Significant Shift (Action Required)
9,open_acc,NaN,Significant Shift (Action Required)
